In [8]:
import pandas as pd 
import numpy as np
from datetime import datetime, date

In [9]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
    version="v2",
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

https://api.allenneuraldynamics.org/v2/metadata_index/data_assets


In [12]:
aggregate = [
  {
    "$match": {
      "data_description.project_name": "V1 Deep Dive", 
      "name": {"$regex": "filtered"},
      "location": {"$regex": "aind-open-data"}, 
      }
  },
  {
    "$project": {
      "name": 1, 
      "subject_id": "$data_description.subject_id",
      "genotype": "$subject.subject_details.genotype", 
      "date_of_birth": "$subject.subject_details.date_of_birth", 
      "sex": "$subject.subject_details.sex", 
      "session_start_time": "$acquisition.acquisition_start_time",
      "session_end_time": "$acquisition.acquisition_end_time",
      "project_name": "$data_description.project_name", 
      "modality": "$data_description.modalities.name",
      "column": { "$arrayElemAt": ["$data_description.tags", 0] },
      "volume": { "$arrayElemAt": ["$data_description.tags", 1] }, 
      "depth": "$acquisition.data_streams.configurations.images.planes.depth",
      "targeted_structure": "$acquisition.data_streams.configurations.images.planes.targeted_structure.acronym"

    }
  },
]
    
records = docdb_api_client.aggregate_docdb_records(
    pipeline = aggregate,
)

In [13]:
df = pd.DataFrame(records)

df['session_date'] = df.apply(lambda x: datetime.fromisoformat(x['session_start_time']).date(), axis=1)
df['session_start_time'] = df.apply(lambda x: datetime.fromisoformat(x['session_start_time']).time(), axis=1)
df['session_end_time'] = df.apply(lambda x: datetime.fromisoformat(x['session_end_time']).time(), axis=1)
df['date_of_birth'] = df.apply(lambda x: datetime.strptime(x['date_of_birth'], '%Y-%m-%d').date(), axis=1)
df['age'] = df.apply(lambda x: (x['session_date'] - x['date_of_birth']).days, axis=1)

df['column'] = df.apply(lambda x: int(x['column'].split(' ')[-1]), axis=1)
df['volume'] = df.apply(lambda x: int(x['volume'].split(' ')[-1]), axis=1)
df['depth'] = df.apply(lambda x: list(np.array(x['depth']).flatten()), axis=1)
df['targeted_structure'] = df.apply(lambda x: np.unique(list(np.array(records[0]['targeted_structure']).flatten()))[0], axis=1)


df['golden_mouse'] = False
df.loc[df.subject_id=='409828', 'golden_mouse'] = True

order = ['project_name','_id','name','subject_id','golden_mouse','genotype','date_of_birth','age', 'sex','modality',
         'session_date','session_start_time', 'session_end_time','column','volume', 'depth', 'targeted_structure']

df = df[order].sort_values(by=['subject_id', 'column', 'volume'])
df

,project_name,_id,name,subject_id,golden_mouse,genotype,date_of_birth,age,sex,modality,session_date,session_start_time,session_end_time,column,volume,depth,targeted_structure
27,V1 Deep Dive,5b6b926a-0771-4664-8053-c52fa0e56189,409828_2018-12-12_12-24-18_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,162,Male,"[Planar optical physiology, Behavior videos]",2018-12-12,12:24:18.963730,13:24:27.772570,1,1,"[130, 114, 98, 82, 66, 50]",VISp
60,V1 Deep Dive,88412a4c-b448-4efb-9825-d785bd5b56ba,409828_2018-12-13_13-21-39_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,163,Male,"[Planar optical physiology, Behavior videos]",2018-12-13,13:21:39.888000,14:21:12.698880,1,2,"[226, 210, 194, 178, 162, 146]",VISp
98,V1 Deep Dive,dac12127-2aed-4392-ae2f-4a57e1bf85e5,409828_2018-12-13_15-10-05_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,163,Male,"[Planar optical physiology, Behavior videos]",2018-12-13,15:10:05.562960,16:09:36.222020,1,3,"[322, 306, 290, 274, 258, 242]",VISp
30,V1 Deep Dive,34d05e9e-ee69-412b-b98d-47dd9d79082e,409828_2018-12-14_13-14-42_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,164,Male,"[Planar optical physiology, Behavior videos]",2018-12-14,13:14:42.634500,14:15:00.029230,1,4,"[418, 402, 386, 370, 354, 338]",VISp
47,V1 Deep Dive,925bfce3-6324-40bc-822a-7e0206717880,409828_2018-12-14_14-47-35_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,164,Male,"[Planar optical physiology, Behavior videos]",2018-12-14,14:47:35.097180,15:46:50.098970,1,5,"[514, 498, 482, 466, 450, 434]",VISp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14,V1 Deep Dive,7854b321-f941-4df9-8a4a-5c27d0cdfd4e,438833_2019-03-21_14-08-14_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,110,Male,"[Planar optical physiology, Behavior videos]",2019-03-21,14:08:14.261290,15:10:53.254690,5,1,"[130, 114, 98, 82, 66, 50]",VISp
75,V1 Deep Dive,1d8814a4-37f0-4367-95ef-e293b9ea5825,438833_2019-03-27_11-33-05_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,116,Male,"[Planar optical physiology, Behavior videos]",2019-03-27,11:33:05.464370,12:31:50.157580,5,2,"[226, 210, 194, 178, 162, 146]",VISp
90,V1 Deep Dive,06219616-2a8e-4693-87b5-98f484914ac6,438833_2019-03-27_14-24-07_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,116,Male,"[Planar optical physiology, Behavior videos]",2019-03-27,14:24:07.041320,15:23:46.589260,5,3,"[322, 306, 290, 274, 258, 242]",VISp
65,V1 Deep Dive,9cb947f6-d632-4f71-a0af-5cfad535bcdf,438833_2019-04-01_14-39-42_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,121,Male,"[Planar optical physiology, Behavior videos]",2019-04-01,14:39:42.230830,15:39:10.801820,5,4,"[418, 402, 386, 370, 354, 338]",VISp


In [16]:
df.to_csv('/data/metadata/V1DD_metadata.csv', index= False)